# ViZDoom Deathmatch — MARL com 6 Agentes (Self-Play + Shared Learner)

## Visão geral do projeto

Este é o **projeto final** de aprendizado por reforço multiagente (MARL) jogando *Doom Deathmatch*. São **6 agentes neurais** que disputam a mesma partida em rede contra **2 bots nativos** do Doom (o mapa `cig` tem 8 slots — 6 + 2 = 8, o máximo). Cada agente:

- roda em seu **próprio processo** com sua **própria instância ViZDoom**,
- enxerga a arena apenas pela **sua visão em primeira pessoa** (POV),
- envia transições para um **único learner centralizado** (Shared Learner),
- recebe periodicamente os **pesos atualizados** da política compartilhada.

É o paradigma clássico de **Centralized Training, Decentralized Execution (CTDE)** com **parameter sharing** — justificado porque todos os agentes são homogêneos (mesma observação, mesmo espaço de ação, mesma recompensa). Por serem 6 agentes coletando em paralelo, o learner recebe ~6× mais experiência por segundo de wall-clock.

## Algoritmo: Rainbow-like Distributional Dueling DQN

Optei por **DQN distribucional** ao invés de PPO porque DQN é **off-policy** e funciona perfeitamente com o esquema de N processos alimentando um único *replay buffer* — PPO exigiria coleta sincronizada estrita. A configuração final combina vários componentes de SOTA do paper *Rainbow* (Hessel et al., 2018):

| Componente | Por que |
|---|---|
| **C51 Distributional** | Aprende a distribuição completa do retorno (51 átomos), não apenas a média — captura risco e multimodalidade |
| **Dueling Network** | Separa valor V(s) e vantagem A(s,a); aprende quais estados são valiosos *independentemente* da ação |
| **Double DQN** | A rede online escolhe a ação; a rede target avalia — remove o viés de superestimação |
| **Noisy Networks** | Exploração parametrizada via ruído nos pesos das camadas finais — substitui ε-greedy de forma adaptativa |
| **Prioritized Experience Replay** | Amostra transições com maior erro TD com mais frequência (sum-tree, α=0.5, β: 0.4→1.0) |
| **5-step returns** | Reduz o viés do bootstrapping de 1 passo, com pouca variância extra |
| **Mixed Precision (AMP)** | Treino em FP16/BF16 com `GradScaler` — ~2× mais rápido na T4 |
| **AdamW + weight decay** | Melhor regularização que Adam puro |
| **League snapshots** | Workers ocasionalmente carregam *snapshots antigos* da política como adversários estáveis — combate o problema de alvo móvel em self-play |

## Arquitetura

- **Entrada**: pilha de 4 frames em escala de cinza (84×84), pré-processados
- **Trunk convolucional**: 3 camadas conv (32→64→64) tipo Atari/DQN
- **Heads dueling**: dois ramos (valor e vantagem) com `NoisyLinear` nas duas últimas camadas
- **Saída**: distribuição categórica de 51 átomos por ação, ∈ [-10, +20]
- **Ações**: 14 ações discretas (movimento, rotação, strafe, ataque + combinações)

## Recompensa shaping

Frag domina, mas várias dimensões auxiliares aceleram o aprendizado:

- `+1.0` por frag
- `+0.01` por ponto de dano causado, `-0.005` por dano recebido
- `+0.02` por hit confirmado, `+0.05` por item coletado
- `+0.001` por ponto de vida ganho, `-0.5` ao morrer
- recompensa clampada em [-3, +5] para estabilidade do C51

## O que você obtém

Ao final do treinamento: **um MP4 por agente** (POV de cada um no seu melhor episódio), o modelo final treinado, e snapshots intermediários para análise.

> **Antes de rodar**: `Runtime → Change runtime type → T4 GPU`. Treinar 2M passos do host (~12M de experiência total) leva algumas horas na T4.

> O script completo (sem comentários) está em `doom_deathmatch.py`. Este notebook apenas o importa, configura e exibe resultados.

## 1. Instalação de dependências

ViZDoom traz wheels pré-compiladas e o mapa `cig.cfg` (deathmatch multiplayer) embutido.

In [ ]:
%%bash
set +e
apt-get update -qq
apt-get install -y -qq ffmpeg libsdl2-2.0-0 libopenal1 > /dev/null
pip install -q --upgrade pip
pip install -q vizdoom
pip install -q opencv-python-headless imageio "imageio-ffmpeg>=0.4.9"

python - <<'PY'
import vizdoom, os
print('vizdoom', getattr(vizdoom, '__version__', '?'))
cig = os.path.join(vizdoom.scenarios_path, 'cig.cfg')
print('cig.cfg presente:', os.path.isfile(cig))
PY
echo 'dependencias instaladas.'

## 2. Baixar o script de treinamento

Toda a lógica (env, rede C51 Dueling Noisy, Prioritized Replay com SumTree, learner, workers, loop de treino) está em `doom_deathmatch.py`. Suba o arquivo para o Colab (ícone de pasta → upload) ou monte o Drive onde ele está. Aqui apenas o copiamos para `/content/` e importamos.

**Opção A** — Upload manual: clique no ícone de pasta à esquerda no Colab, arraste `doom_deathmatch.py` para `/content/`.

**Opção B** — Drive: descomente o bloco abaixo se o script estiver no seu Drive.

In [ ]:
import os, shutil

# Opcao B: descomente para puxar do Drive
# from google.colab import drive
# drive.mount('/content/drive')
# shutil.copy('/content/drive/MyDrive/doom_deathmatch.py', '/content/doom_deathmatch.py')

assert os.path.isfile('/content/doom_deathmatch.py'), \
    'Faca upload de doom_deathmatch.py para /content/ antes de continuar.'
print('Script encontrado em /content/doom_deathmatch.py')

## 3. Configuração da partida

Os parâmetros principais. **Não mude** `N_PLAYERS=6` e `N_BOTS=2` (= 8 slots do mapa `cig`). `TOTAL_STEPS` é o número de passos do *host* — a experiência total coletada pelo learner é ~6× isso pois cada agente alimenta o buffer.

In [ ]:
USE_DRIVE = False

SCENARIO        = 'cig'
N_PLAYERS       = 6
N_BOTS          = 2
PORT            = 5029
TOTAL_STEPS     = 2_000_000
EPISODE_MINUTES = 3.0
SEED            = 42

BUFFER_SIZE     = 100_000
BATCH_SIZE      = 64
LR              = 6.25e-5
N_STEP          = 5
GAMMA           = 0.99
LEARNING_STARTS = 20_000
TARGET_SYNC     = 8_000
SNAPSHOT_EVERY  = 100_000

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/doom_deathmatch'
else:
    SAVE_DIR = '/content/doom_deathmatch'

import os
os.makedirs(SAVE_DIR, exist_ok=True)
assert N_PLAYERS + N_BOTS <= 8, 'O mapa cig tem 8 slots no maximo.'
print(f'{N_PLAYERS} agentes aprendizes + {N_BOTS} bots | cenario={SCENARIO}')
print(f'TOTAL_STEPS (host) = {TOTAL_STEPS:,}  (experiencia total ~ {TOTAL_STEPS*N_PLAYERS:,})')
print('Artefatos em:', SAVE_DIR)

## 4. Importar o módulo e validar GPU

O script funciona como módulo Python. Importamos `train` e funções auxiliares dele. Sem GPU o treinamento é inviável — confirme T4.

In [ ]:
import sys
if '/content' not in sys.path:
    sys.path.insert(0, '/content')

import importlib
import doom_deathmatch
importlib.reload(doom_deathmatch)
from doom_deathmatch import train, bundle_results

import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('AVISO: sem GPU. Vai treinar muito devagar — troque o runtime para T4.')

## 5. Treinamento

Aqui é onde tudo acontece. O fluxo é:

1. **Spawn dos 5 workers** (rank 1 a 5) *antes* de tocar em CUDA — processos filhos limpos, inferência em CPU.
2. **Host (agent 0) abre a partida** com `+host 6 -port 5029 -deathmatch` e adiciona os 2 bots.
3. Os workers fazem `-join` e a partida começa em lockstep (sincronia 35 tics/s).
4. Cada agente coleta sua POV, monta retornos de 5 passos via `NStep`, e envia transições para `transition_q`.
5. O host drena a fila a cada iteração e empurra tudo no `PrioritizedReplay` (sum-tree).
6. O learner faz um `train_step` por iteração: amostra prioritizada → projeção C51 → cross-entropy → update via AMP.
7. A cada `publish_every=1500` passos os pesos atualizados são gravados em `policy_latest.pt` (escrita atômica).
8. A cada `snapshot_every=100k` passos uma cópia versionada é gravada em `snapshots/snapshot_NNNN.pt`.
9. Workers com `snapshot_prob > 0` ocasionalmente recarregam um snapshot antigo no início do episódio, criando *adversários estáveis* (league-style).
10. Cada agente mantém em disco `agent<rank>_best.mp4` — sua POV do melhor episódio até agora.

**Robustez**: processos ViZDoom *stale* são mortos na entrada (`pkill`), e o `finally` garante limpeza dos workers. Se o treino travar: `Runtime → Interrupt` e re-execute esta célula.

Para retomar: passe `resume=os.path.join(SAVE_DIR, 'deathmatch_latest.pt')`.

In [ ]:
FINAL_MODEL = train(
    scenario=SCENARIO,
    n_players=N_PLAYERS,
    n_bots=N_BOTS,
    port=PORT,
    total_steps=TOTAL_STEPS,
    save_dir=SAVE_DIR,
    episode_minutes=EPISODE_MINUTES,
    n_step=N_STEP,
    gamma=GAMMA,
    buffer_size=BUFFER_SIZE,
    batch_size=BATCH_SIZE,
    lr=LR,
    learning_starts=LEARNING_STARTS,
    target_sync_every=TARGET_SYNC,
    save_every=50_000,
    log_every=1_000,
    publish_every=1_500,
    snapshot_every=SNAPSHOT_EVERY,
    seed=SEED,
    amp=True,
    # resume=f'{SAVE_DIR}/deathmatch_latest.pt',  # descomente p/ continuar
)
print('Modelo final:', FINAL_MODEL)

## 6. Assistir cada agente

Um vídeo por agente — POV de primeira pessoa do **melhor episódio** (mais frags) de cada um. Como cada agente grava sua própria melhor partida em disco, há vídeos utilizáveis mesmo se você interromper o treino antes do final.

In [ ]:
from IPython.display import HTML, display
import base64, glob, os

paths = sorted(glob.glob(os.path.join(SAVE_DIR, 'agent*_best.mp4')))
if not paths:
    print('Nenhum video ainda — deixe o treino terminar pelo menos um episodio.')
else:
    blocks = []
    for p in paths:
        name = os.path.basename(p).replace('_best.mp4', '').upper()
        with open(p, 'rb') as f:
            b64 = base64.b64encode(f.read()).decode()
        blocks.append(
            '<div style="display:inline-block;margin:6px;text-align:center;'
            'vertical-align:top">'
            f'<div style="font:600 13px sans-serif;padding:4px">{name} — POV</div>'
            '<video width=360 controls autoplay loop muted>'
            f'<source src="data:video/mp4;base64,{b64}" type="video/mp4">'
            '</video></div>')
    display(HTML('<div style="white-space:normal">' + ''.join(blocks) + '</div>'))
    print(f'Exibindo {len(paths)} videos:',
          ', '.join(os.path.basename(p) for p in paths))

## 7. Download dos resultados

Empacota todos os vídeos dos 6 agentes + o modelo final treinado em um zip (pulado se você usou Drive — já está salvo lá).

In [ ]:
if not USE_DRIVE:
    from google.colab import files
    bundle = bundle_results(SAVE_DIR)
    print('Bundle:', bundle)
    files.download(bundle)
else:
    print('USE_DRIVE=True — artefatos ja estao no Drive em', SAVE_DIR)

## Notas finais sobre o design MARL

**Por que Shared Learner + parameter sharing (e não Independent Learners)?**  
Como todos os agentes são homogêneos (mesma observação, mesma ação, mesma recompensa), compartilhar pesos é estritamente melhor: cada agente *vê* mais dados durante o treino, e generaliza para o papel de qualquer um dos 6 jogadores. Independent Learners seriam justificáveis apenas se os agentes tivessem papéis assimétricos (ex.: predador vs. presa).

**Por que não PPO / MAPPO?**  
PPO é on-policy: precisa de coleta sincronizada para preservar a validade da estimativa de vantagem. O esquema multiprocesso ViZDoom com 6 clientes em lockstep funciona, mas a latência de sincronia inter-processo limita o throughput. DQN off-policy aproveita 100% da experiência coletada por todos os agentes (mesmo dados *staleness*) sem perda teórica — exatamente o cenário ideal para o setup Ape-X / R2D2 / Rainbow.

**Por que league snapshots em self-play?**  
Self-play puro tem o problema de *alvo móvel*: enquanto o agente A aprende a derrotar B, B também muda. Isso gera ciclos de estratégia ao invés de convergência para um Nash robusto. Carregar ocasionalmente snapshots antigos (~30% das vezes em ranks altos) cria adversários *congelados* — o agente atual precisa derrotar não só a si mesmo de agora, mas a si mesmo *de várias épocas anteriores*, forçando estratégias robustas. É a essência simplificada do *AlphaStar League*.

**Por que C51 distribucional?**  
Deathmatch tem retornos altamente multimodais — sair vencendo ou sendo dizimado têm probabilidades não-triviais. A média (Q-learning convencional) joga fora essa informação. C51 aprende a distribuição completa em 51 átomos no intervalo [-10, +20], o que (a) melhora a estabilidade do gradiente, (b) acelera convergência em ambientes esparso-recompensados, e (c) permite políticas sensíveis a risco se quisermos.

**Limites do mapa `cig`**  
8 slots (= `N_PLAYERS + N_BOTS`). Para mais agentes, seria necessário um mapa custom (`.wad`). Para deathmatch competitivo padrão, 6 humanos + 2 bots é uma escala excelente — densidade de combate alta sem caos absoluto.

**Tempo de treino esperado**  
Em uma T4 do Colab, ~6-10 horas para 2M passos do host. Frags por episódio começam negativos (mortes > frags) e cruzam zero em ~300k passos, estabilizando em 8-15 frags/episódio na fase final, dependendo da partida e do oponente snapshot escolhido.